# TML26 Stolen Model Detection
# team_XI

## 1. Setup

In [ ]:
import os, gc, json, math, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, Dataset

from torchvision import datasets, transforms
from torchvision.models import resnet18

try:
    from safetensors.torch import load_file
except Exception:
    !pip -q install safetensors
    from safetensors.torch import load_file

try:
    from huggingface_hub import snapshot_download
except Exception:
    !pip -q install huggingface_hub
    from huggingface_hub import snapshot_download

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

ROOT = Path('/kaggle/working/tml26_task2')
ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT = Path('/kaggle/working/data')
FEATURE_V2_PATH = ROOT / 'features_v2.csv'

## 2. Download

In [ ]:
HF_REPO = 'SprintML/tml26_task2'
HF_LOCAL_DIR = ROOT / 'hf_repo'

if not (HF_LOCAL_DIR / 'target_model' / 'weights.safetensors').exists():
    snapshot_download(repo_id=HF_REPO, repo_type='model',
                      local_dir=str(HF_LOCAL_DIR), local_dir_use_symlinks=False)

TARGET_PATH = HF_LOCAL_DIR / 'target_model' / 'weights.safetensors'
TRAIN_IDX_PATH = HF_LOCAL_DIR / 'target_model' / 'train_main_idx.json'
SUSPECT_DIR = HF_LOCAL_DIR / 'suspect_models'
suspect_paths = sorted(SUSPECT_DIR.glob('suspect_*.safetensors'))
print('target exists:', TARGET_PATH.exists(), '| num suspects:', len(suspect_paths))

## 3. Model loading

In [ ]:
def make_model():
    m = resnet18(weights=None)
    m.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc = nn.Linear(m.fc.in_features, 100)
    return m

def load_state(path):
    return load_file(str(path), device='cpu')

def load_model(path, device=DEVICE):
    sd = load_state(path)
    m = make_model()
    m.load_state_dict(sd, strict=True)
    m.to(device).eval()
    return m

target_model = load_model(TARGET_PATH)
target_state = load_state(TARGET_PATH)
print('target loaded')

## 4. Probe sets

In [ ]:
MEAN = (0.5071, 0.4867, 0.4408)
STD  = (0.2675, 0.2565, 0.2761)
base_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

train_full = datasets.CIFAR100(root=str(DATA_ROOT), train=True, download=True, transform=base_tf)
test_full  = datasets.CIFAR100(root=str(DATA_ROOT), train=False, download=True, transform=base_tf)

with open(TRAIN_IDX_PATH, 'r') as f:
    target_train_idx = json.load(f)

rng = np.random.default_rng(SEED)
N_TARGET_TRAIN = 1000  
N_TEST         = 1000
N_JAC          = 64   

probe_train_idx = rng.choice(target_train_idx, size=N_TARGET_TRAIN, replace=False).tolist()
probe_test_idx  = rng.choice(np.arange(len(test_full)), size=N_TEST, replace=False).tolist()

# Pre-materialize the CutMix probe ONCE
cm_loader = DataLoader(Subset(train_full, probe_train_idx), batch_size=256, shuffle=False, num_workers=2)
cm_x = torch.cat([x for x, _ in cm_loader], dim=0)

# Deterministic center-cut CutMix
def make_cutmix(x, seed=42, lam=0.5):
    g = torch.Generator().manual_seed(seed)
    n, _, h, w = x.shape
    perm = torch.randperm(n, generator=g)
    cut_rat = math.sqrt(1.0 - lam)
    cw, ch = int(w * cut_rat), int(h * cut_rat)
    cx, cy = w // 2, h // 2
    x1, x2 = max(cx - cw // 2, 0), min(cx + cw // 2, w)
    y1, y2 = max(cy - ch // 2, 0), min(cy + ch // 2, h)
    out = x.clone()
    out[:, :, y1:y2, x1:x2] = x[perm, :, y1:y2, x1:x2]
    return out

cutmix_x = make_cutmix(cm_x, seed=42, lam=0.5)
cutmix_y = torch.zeros(len(cutmix_x), dtype=torch.long)  # labels unused

class TensorDS(Dataset):
    def __init__(self, x, y): self.x, self.y = x, y
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i], self.y[i]

loaders = {
    'target_train': DataLoader(Subset(train_full, probe_train_idx), batch_size=256, shuffle=False, num_workers=2),
    'test':         DataLoader(Subset(test_full, probe_test_idx),  batch_size=256, shuffle=False, num_workers=2),
    'cutmix':       DataLoader(TensorDS(cutmix_x, cutmix_y), batch_size=256, shuffle=False, num_workers=0),
}

# Jacobian probe
jac_loader = DataLoader(Subset(test_full, probe_test_idx[:N_JAC]), batch_size=N_JAC, shuffle=False, num_workers=0)
jac_x = next(iter(jac_loader))[0]

for k, v in loaders.items():
    print(k, len(v.dataset))
print('jac probe:', jac_x.shape)

## 5. Feature functions

In [ ]:

def cos_np(a, b):
    a = np.asarray(a, dtype=np.float64).ravel()
    b = np.asarray(b, dtype=np.float64).ravel()
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

def spearman(a, b):
    a = np.asarray(a, dtype=np.float64).ravel()
    b = np.asarray(b, dtype=np.float64).ravel()
    if a.size < 2: return 0.0
    ra = a.argsort().argsort().astype(np.float64)
    rb = b.argsort().argsort().astype(np.float64)
    ra -= ra.mean(); rb -= rb.mean()
    den = np.linalg.norm(ra) * np.linalg.norm(rb) + 1e-12
    return float(np.dot(ra, rb) / den)

#forward pass that returns penultimate feat
@torch.no_grad()
def collect_outputs(model, loaders):
    out = {}
    model.eval()
    for name, loader in loaders.items():
        logits_l, feat_l, lab_l = [], [], []
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            f = model.conv1(x); f = model.bn1(f); f = model.relu(f); f = model.maxpool(f)
            f = model.layer1(f); f = model.layer2(f); f = model.layer3(f); f = model.layer4(f)
            f = model.avgpool(f); feat = torch.flatten(f, 1)
            logits = model.fc(feat)
            logits_l.append(logits.cpu().float())
            feat_l.append(feat.cpu().float())
            lab_l.append(y)
        logits = torch.cat(logits_l).numpy()
        feat   = torch.cat(feat_l).numpy()
        labels = torch.cat(lab_l).numpy()
        probs  = torch.softmax(torch.from_numpy(logits), dim=1).numpy()
        top2   = np.sort(logits, axis=1)[:, -2:]
        margin = top2[:, 1] - top2[:, 0]
        out[name] = dict(
            logits=logits.astype(np.float32),
            feat=feat.astype(np.float32),
            probs=probs.astype(np.float32),
            pred=logits.argmax(axis=1).astype(np.int64),
            margin=margin.astype(np.float32),
            labels=labels.astype(np.int64),
        )
    return out

#SAC: sample correlation
def sample_correlation_matrix(X):
    X = X.astype(np.float64)
    X = X - X.mean(axis=1, keepdims=True)
    n = np.linalg.norm(X, axis=1, keepdims=True) + 1e-12
    Xn = X / n
    return Xn @ Xn.T

def sac_score(t_out, s_out):
    Ct = sample_correlation_matrix(t_out)
    Cs = sample_correlation_matrix(s_out)
    iu = np.triu_indices(Ct.shape[0], k=1)
    vt, vs = Ct[iu], Cs[iu]
    return float(np.dot(vt, vs) / (np.linalg.norm(vt) * np.linalg.norm(vs) + 1e-12))

# Linear CKA on penultimate features
def linear_cka(X, Y):
    X = X.astype(np.float64); Y = Y.astype(np.float64)
    X -= X.mean(axis=0, keepdims=True); Y -= Y.mean(axis=0, keepdims=True)
    xy = np.linalg.norm(X.T @ Y, 'fro') ** 2
    xx = np.linalg.norm(X.T @ X, 'fro')
    yy = np.linalg.norm(Y.T @ Y, 'fro')
    return float(xy / (xx * yy + 1e-12))

#Input-Jacobian cosine
def input_grad(model, x, target_class):
    x = x.detach().clone().requires_grad_(True)
    logits = model(x)
    sel = logits.gather(1, target_class.view(-1, 1)).sum()
    return torch.autograd.grad(sel, x)[0].detach()

def jacobian_cosine(t_model, s_model, probe_x, bs=32):
    t_model.eval(); s_model.eval()
    tot = 0.0; n = 0
    for i in range(0, probe_x.size(0), bs):
        xb = probe_x[i:i+bs].to(DEVICE)
        with torch.no_grad():
            ref = t_model(xb).argmax(dim=1)
        gt = input_grad(t_model, xb, ref).flatten(1)
        gs = input_grad(s_model, xb, ref).flatten(1)
        tot += float(F.cosine_similarity(gt, gs, dim=1).sum().item())
        n += xb.size(0)
    return tot / max(n, 1)

#BN running stats
def bn_stats_cos(t_state, s_state):
    rmt, rms, rvt, rvs = [], [], [], []
    for k, t in t_state.items():
        if not isinstance(t, torch.Tensor): continue
        if 'running_mean' in k:
            rmt.append(t.float().flatten()); rms.append(s_state[k].float().flatten())
        elif 'running_var' in k:
            rvt.append(t.float().flatten()); rvs.append(s_state[k].float().flatten())
    out = {}
    if rmt:
        tm = torch.cat(rmt).numpy(); sm = torch.cat(rms).numpy()
        out['bn_runmean_cos']    = cos_np(tm, sm)
        out['bn_runmean_l1_neg'] = -float(np.mean(np.abs(tm - sm)))
    if rvt:
        tv = torch.cat(rvt).numpy(); sv = torch.cat(rvs).numpy()
        out['bn_runvar_logcos']  = cos_np(np.log1p(tv), np.log1p(sv))
    return out

print('feature functions ready')

## 6. Cache target outputs

In [ ]:
target_out = collect_outputs(target_model, loaders)
for k, v in target_out.items():
    print(k, v['logits'].shape)

## 7. Score every suspect

In [ ]:
rows = []
done_ids = set()
if FEATURE_V2_PATH.exists():
    old = pd.read_csv(FEATURE_V2_PATH)
    rows = old.to_dict('records')
    done_ids = set(old['id'].astype(int).tolist())
    print('resuming, already done:', len(done_ids))

for path in suspect_paths:
    sid = int(path.stem.split('_')[-1])
    if sid in done_ids:
        continue
    print(f'scoring suspect {sid:03d}', end=' ')
    suspect_state = load_state(path)
    suspect_model = make_model().to(DEVICE)
    suspect_model.load_state_dict(suspect_state, strict=True)
    suspect_model.eval()

    suspect_out = collect_outputs(suspect_model, loaders)

    row = {'id': sid}
    row.update(bn_stats_cos(target_state, suspect_state))
    for name in loaders:
        row[f'{name}_sac']             = sac_score(target_out[name]['probs'], suspect_out[name]['probs'])
        row[f'{name}_cka']             = linear_cka(target_out[name]['feat'], suspect_out[name]['feat'])
        row[f'{name}_margin_spearman'] = spearman(target_out[name]['margin'], suspect_out[name]['margin'])
        row[f'{name}_pred_agree']      = float((target_out[name]['pred'] == suspect_out[name]['pred']).mean())
    row['jacobian_cos'] = jacobian_cosine(target_model, suspect_model, jac_x)

    rows.append(row)
    pd.DataFrame(rows).sort_values('id').to_csv(FEATURE_V2_PATH, index=False)
    print('done')

    del suspect_model, suspect_state, suspect_out
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

features = pd.read_csv(FEATURE_V2_PATH).sort_values('id').reset_index(drop=True)
print('shape:', features.shape)
features.head()

## 8. Build all score variants

In [ ]:
def robust_z(x):
    x = np.asarray(x, dtype=np.float64)
    med = np.median(x)
    mad = np.median(np.abs(x - med)) + 1e-12
    return (x - med) / (1.4826 * mad)

def to_rank_unit(s):
    return pd.Series(s).rank(method='average', pct=True).values

def zframe(df, cols):
    z = pd.DataFrame(index=df.index)
    for c in cols:
        v = df[c].replace([np.inf, -np.inf], np.nan)
        v = v.fillna(v.median())
        z[c] = robust_z(v.values)
    return z

bn_cols     = [c for c in ['bn_runmean_cos', 'bn_runmean_l1_neg', 'bn_runvar_logcos'] if c in features.columns]
sac_cols    = [c for c in features.columns if c.endswith('_sac')]
cka_cols    = [c for c in features.columns if c.endswith('_cka')]
margin_cols = [c for c in features.columns if c.endswith('_margin_spearman')]
pred_cols   = [c for c in features.columns if c.endswith('_pred_agree')]
jac_cols    = ['jacobian_cos'] if 'jacobian_cos' in features.columns else []

z_bn     = zframe(features, bn_cols)     if bn_cols else pd.DataFrame(index=features.index)
z_sac    = zframe(features, sac_cols)    if sac_cols else pd.DataFrame(index=features.index)
z_cka    = zframe(features, cka_cols)    if cka_cols else pd.DataFrame(index=features.index)
z_margin = zframe(features, margin_cols) if margin_cols else pd.DataFrame(index=features.index)
z_pred   = zframe(features, pred_cols)   if pred_cols else pd.DataFrame(index=features.index)
z_jac    = zframe(features, jac_cols)    if jac_cols else pd.DataFrame(index=features.index)
z_all    = pd.concat([z_bn, z_sac, z_cka, z_margin, z_pred, z_jac], axis=1)

def mean_or_zero(z): return z.mean(axis=1).values if not z.empty else np.zeros(len(features))
bn_s, sac_s, cka_s, marg_s, jac_s = (mean_or_zero(z) for z in (z_bn, z_sac, z_cka, z_margin, z_jac))

variants = {
    # single-signal probes
    'bn_only':       bn_s,
    'sac_only':      sac_s,
    'cka_only':      cka_s,
    'jacobian_only': jac_s,

    #aggregators over all v2 features
    'max':       z_all.values.max(axis=1),                             
    'top3_mean': np.sort(z_all.values, axis=1)[:, -3:].mean(axis=1),   
    'top5_mean': np.sort(z_all.values, axis=1)[:, -5:].mean(axis=1), 

    # hand-weighted ensembles
    'ensemble':  0.45 * bn_s + 0.25 * sac_s + 0.20 * cka_s + 0.10 * jac_s,
    'bn_or_func': np.maximum(bn_s, np.max(np.stack([sac_s, cka_s, jac_s]), axis=0)),
}

# Save all submissions
for name, raw in variants.items():
    sub = pd.DataFrame({'id': features['id'].astype(int).values,
                        'score': to_rank_unit(raw)})
    sub.to_csv(ROOT / f'submission_{name}.csv', index=False)

print('saved variants:')
for p in sorted(ROOT.glob('submission_*.csv')): print(' ', p.name)

diag = pd.DataFrame({'id': features['id'].astype(int).values})
for name, raw in variants.items():
    diag[name] = to_rank_unit(raw)
print('\nTop 10 by max-of-z:')
print(diag.sort_values('max', ascending=False).head(10).to_string(index=False))

In [ ]:

# Extra signals: boundary probes + high-confidence SAC
# Adds two new feature groups per suspect.
import torch, gc
import numpy as np
import pandas as pd
import torch.nn.functional as F

EXTRAS_PATH = ROOT / 'features_v2_extras.csv'

def make_boundary_probes(target_model, base_x, eps=8/255, alpha=2/255, steps=20):
    target_model.eval()
    x = base_x.clone().to(DEVICE)
    with torch.no_grad():
        top2 = target_model(x).topk(2, dim=1).indices
        target_class = top2[:, 1]
    x_adv = x.clone().detach()
    for _ in range(steps):
        x_adv.requires_grad_(True)
        logits = target_model(x_adv)
        loss = F.cross_entropy(logits, target_class)
        grad = torch.autograd.grad(loss, x_adv)[0]
        x_adv = (x_adv.detach() - alpha * grad.sign())
        x_adv = torch.max(torch.min(x_adv, x + eps), x - eps)
    return x_adv.detach().cpu()

boundary_x = make_boundary_probes(target_model, cm_x[:256])
print('boundary probes shape:', boundary_x.shape)

# Target predictions on boundary probes
with torch.no_grad():
    t_logits_b = target_model(boundary_x.to(DEVICE)).cpu()
t_pred_b   = t_logits_b.argmax(dim=1).numpy()
t_top2_b   = torch.sort(t_logits_b, dim=1, descending=True).values[:, :2].numpy()
t_margin_b = t_top2_b[:, 0] - t_top2_b[:, 1]
t_probs_b  = torch.softmax(t_logits_b, dim=1).numpy()

# High-confidence indices on target_train
t_margins_train = target_out['target_train']['margin']
hi_conf_idx = np.argsort(t_margins_train)[-300:]
t_probs_hi  = target_out['target_train']['probs'][hi_conf_idx]
print(f'high-confidence samples kept: {len(hi_conf_idx)}')

extras_rows = []
done_ids = set()
if EXTRAS_PATH.exists():
    old = pd.read_csv(EXTRAS_PATH)
    extras_rows = old.to_dict('records')
    done_ids = set(old['id'].astype(int).tolist())
    print(f'resuming, already done: {len(done_ids)}')

for path in suspect_paths:
    sid = int(path.stem.split('_')[-1])
    if sid in done_ids:
        continue
    print(f'extras for {sid:03d}', end=' ')
    suspect_state = load_state(path)
    suspect_model = make_model().to(DEVICE)
    suspect_model.load_state_dict(suspect_state, strict=True)
    suspect_model.eval()

    # Boundary probe forward
    with torch.no_grad():
        s_logits_b = suspect_model(boundary_x.to(DEVICE)).cpu()
    s_pred_b   = s_logits_b.argmax(dim=1).numpy()
    s_top2_b   = torch.sort(s_logits_b, dim=1, descending=True).values[:, :2].numpy()
    s_margin_b = s_top2_b[:, 0] - s_top2_b[:, 1]
    s_probs_b  = torch.softmax(s_logits_b, dim=1).numpy()

    # High-confidence SAC
    s_out_train = collect_outputs(suspect_model, {'target_train': loaders['target_train']})
    s_probs_hi  = s_out_train['target_train']['probs'][hi_conf_idx]

    extras_rows.append({
        'id': sid,
        'boundary_agree':           float((t_pred_b == s_pred_b).mean()),
        'boundary_margin_spearman': spearman(t_margin_b, s_margin_b),
        'boundary_sac':             sac_score(t_probs_b, s_probs_b),
        'hi_conf_sac':              sac_score(t_probs_hi, s_probs_hi),
    })
    pd.DataFrame(extras_rows).sort_values('id').to_csv(EXTRAS_PATH, index=False)
    print('done')

    del suspect_model, suspect_state, s_out_train
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

extras = pd.read_csv(EXTRAS_PATH).sort_values('id').reset_index(drop=True)
print('extras shape:', extras.shape)
print(extras.head())
print(extras.describe())

In [ ]:
# Merge extras into the main features and build max_v2
features_full = features.merge(extras, on='id', how='left')

# Z-score the new features
z_boundary = zframe(features_full, ['boundary_agree', 'boundary_margin_spearman', 'boundary_sac'])
z_hi_sac   = zframe(features_full, ['hi_conf_sac'])

bn_best       = z_bn.values.max(axis=1)
sac_best      = z_sac.values.max(axis=1)
cka_best      = z_cka.values.max(axis=1)
jac_best      = z_jac.values.max(axis=1)
boundary_best = z_boundary.values.max(axis=1)
hi_sac_best   = z_hi_sac.values.max(axis=1)

family_max_v2 = np.max(np.stack([bn_best, sac_best, cka_best, jac_best,
                                 boundary_best, hi_sac_best]), axis=0)

all_features_z = pd.concat([z_bn, z_sac, z_cka, z_margin, z_pred, z_jac,
                            z_boundary, z_hi_sac], axis=1)
max_all_v2 = all_features_z.values.max(axis=1)

# Boundary-augmented original max
max_plus_boundary = np.maximum(z_all.values.max(axis=1), np.maximum(boundary_best, hi_sac_best))

for name, raw in [('max_v2', max_all_v2),
                  ('family_max_v2', family_max_v2),
                  ('max_plus_boundary', max_plus_boundary)]:
    sub = pd.DataFrame({'id': features_full['id'].astype(int).values,
                        'score': to_rank_unit(raw)})
    sub.to_csv(ROOT / f'submission_{name}.csv', index=False)

old_top = set(pd.read_csv(ROOT / 'submission_max.csv').sort_values('score', ascending=False).head(30)['id'].astype(int).tolist())
for name in ['max_v2', 'max_plus_boundary']:
    new_top = set(pd.read_csv(ROOT / f'submission_{name}.csv').sort_values('score', ascending=False).head(30)['id'].astype(int).tolist())
    added = new_top - old_top
    removed = old_top - new_top
    print(f'{name}: added {sorted(added)}, removed {sorted(removed)}')

## 9. KNOBS — best variant to submin is max_plus_boundary


In [ ]:

VARIANT = 'max_plus_boundary'       
                      
SUBMIT  = False       
API_KEY = 'YOUR_API_KEY_HERE'

SUBMISSION_PATH = ROOT / f'submission_{VARIANT}.csv'


sub = pd.read_csv(SUBMISSION_PATH)
assert list(sub.columns) == ['id', 'score']
assert len(sub) == 360
assert sub['id'].min() == 0 and sub['id'].max() == 359 and sub['id'].nunique() == 360
assert np.isfinite(sub['score']).all()
print('valid:', SUBMISSION_PATH.name)
print(sub.describe())

if SUBMIT:
    import requests
    BASE_URL = 'http://34.63.153.158'
    TASK_ID  = '19-stolen-model-detection'
    with open(SUBMISSION_PATH, 'rb') as f:
        files = {'file': (SUBMISSION_PATH.name, f, 'csv')}
        resp = requests.post(f'{BASE_URL}/submit/{TASK_ID}',
                             headers={'X-API-Key': API_KEY},
                             files=files, timeout=(10, 120))
    print('status:', resp.status_code)
    try: print(resp.json())
    except Exception: print(resp.text)
else:
    print('SUBMIT=False — set to True when ready')